In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import warnings
from catboost import CatBoostClassifier
import optuna
import joblib
import os
from sklearn.preprocessing import TargetEncoder

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


## **1. Data Loading & EDA**

* We are going to load the dataset and perform an initial exploratory data analysis.
* We will check the structural dimensions and look for any missing values across the columns.
* We plan to identify our target variable and inspect the general format of the data.
* This will give us a clear understanding of the dataset's integrity before any modeling begins.

In [2]:
# Load dataset files
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

In [3]:
# Display basic structural dimensions
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Verify target presence and inspect first rows
print("\nTrain dataset overview:")
print(train_df.head())

print("\nMissing values in train:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

Train shape: (668665, 15)
Test shape: (286571, 14)

Train dataset overview:
   id  Age  Annual_Income_USD  Daily_Commute_km  Number_of_Cars_Owned  \
0   0   66            92887.0              23.4                     2   
1   1   38            30000.0               5.0                     1   
2   2   26            94389.0              36.8                     1   
3   3   66            73580.0              23.7                     2   
4   4   54            57898.0              50.8                     1   

   Charging_Stations_Near_Home  Charging_Stations_Near_Work  \
0                            3                            7   
1                            2                            2   
2                            8                           15   
3                            6                            9   
4                            2                            3   

   Environmental_Concern_Level  Gender City_Type Current_Car_Type  \
0                          1.0    Mal

## **2. Target Transformation & Distribution**

* Following the clean data load, we observed that our target variable is in text format, so we will map it to binary numerical values.
* Once transformed, we are going to analyze the class distribution to check for potential imbalances.
* We will also identify all the remaining categorical columns that require encoding.
* This foundational work ensures our models can process the target correctly and alerts us to the needed evaluation metrics.

In [4]:
# Clean leading/trailing spaces and map target variable
train_df['Will_Buy_EV'] = train_df['Will_Buy_EV'].astype(str).str.strip()
target_map = {'Yes': 1, 'No': 0}
train_df['Will_Buy_EV'] = train_df['Will_Buy_EV'].map(target_map)

# Check target class distribution including potential NaNs
target_distribution = train_df['Will_Buy_EV'].value_counts(normalize=True, dropna=False) * 100
print("Target distribution (%):")
print(target_distribution)

# Identify remaining categorical columns
categorical_cols = train_df.select_dtypes(include=['object', 'str']).columns.tolist()
if 'Will_Buy_EV' in categorical_cols:
    categorical_cols.remove('Will_Buy_EV')

print(f"\nCategorical columns: {categorical_cols}")

Target distribution (%):
Will_Buy_EV
0    82.5355
1    17.4645
Name: proportion, dtype: float64

Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## **3. Categorical Encoding**

* Since we discovered a moderate class imbalance and identified six categorical features, our next move is to transform these text variables into numbers.
* We will apply an `Ordinal Encoder` to convert them efficiently without expanding the dataset's dimensionality.
* We plan to fit this encoder on the training data and then apply it to both the train and test sets.
* This step will leave our dataset fully numerical and ready for algorithmic processing.

In [5]:
# Initialize the encoder handling unseen categories gracefully
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit on train data and transform both train and test sets
train_df[categorical_cols] = encoder.fit_transform(train_df[categorical_cols])
test_df[categorical_cols] = encoder.transform(test_df[categorical_cols])

print("Categorical columns encoded successfully. Here is the new format:")
print(train_df[categorical_cols].head())

Categorical columns encoded successfully. Here is the new format:
   Gender  City_Type  Current_Car_Type  Home_Charging_Possible  \
0     1.0        1.0               2.0                     1.0   
1     1.0        0.0               1.0                     1.0   
2     0.0        2.0               2.0                     0.0   
3     1.0        1.0               0.0                     1.0   
4     1.0        1.0               0.0                     1.0   

   Subsidy_Available  Range_Anxiety_Level  
0                0.0                  1.0  
1                0.0                  1.0  
2                1.0                  1.0  
3                0.0                  1.0  
4                0.0                  1.0  


## **4. Baseline Modeling (LightGBM)**

* Now that all our features are mathematically readable, we will establish our initial performance benchmark.
* We are going to set up a `Stratified K-Fold cross-validation` scheme to respect the class imbalance we found earlier.
* Using this setup, we will train a `LightGBM` model to see how well it can predict the target without any advanced modifications.
* This will provide us with a foundational Out-Of-Fold ROC AUC score to beat in our upcoming optimization phases.

In [6]:
# Define features and target, dropping identifier and target columns
X = train_df.drop(columns=['id', 'Will_Buy_EV'])
y = train_df['Will_Buy_EV']

# Setup Stratified K-Fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))

# Initialize LightGBM classifier
model = lgb.LGBMClassifier(n_estimators=500, random_state=42, n_jobs=-1)

print("Starting cross-validation training...")

# Train model across folds
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Store validation predictions
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold + 1} ROC AUC: {fold_auc:.5f}")

# Calculate and display overall Out-Of-Fold score
oof_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall OOF ROC AUC: {oof_auc:.5f}")

Starting cross-validation training...


c:\Users\USED\Desktop\Trabajo\Data Scientist\Kaggle Competitions\september - predicting electric vehicle purchases\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 93424, number of negative: 441508
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060097 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174646 -> initscore=-1.553048
[LightGBM] [Info] Start training from score -1.553048
Fold 1 ROC AUC: 0.93992


c:\Users\USED\Desktop\Trabajo\Data Scientist\Kaggle Competitions\september - predicting electric vehicle purchases\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016596 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
Fold 2 ROC AUC: 0.94111


...

[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014953 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
Fold 3 ROC AUC: 0.94240


c:\Users\USED\Desktop\Trabajo\Data Scientist\Kaggle Competitions\september - predicting electric vehicle purchases\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012881 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
Fold 4 ROC AUC: 0.94200


c:\Users\USED\Desktop\Trabajo\Data Scientist\Kaggle Competitions\september - predicting electric vehicle purchases\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 619
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
Fold 5 ROC AUC: 0.94109

Overall OOF ROC AUC: 0.94129


## **5. Feature Engineering**

* With our baseline score of `0.94129` established, we will now attempt to improve it by adding domain-specific logic.
* We are going to engineer new composite variables, such as **total charging stations** and **income per car**.
* By giving the model these pre-calculated relationships, we expect to make it easier for the algorithm to find strong predictive patterns.
* We will then retrain our `LightGBM` model on this expanded dataset to evaluate if these new features actually increase our score.

In [7]:
# Aggregate charging station metrics
train_df['Total_Charging_Stations'] = train_df['Charging_Stations_Near_Home'] + train_df['Charging_Stations_Near_Work']
test_df['Total_Charging_Stations'] = test_df['Charging_Stations_Near_Home'] + test_df['Charging_Stations_Near_Work']

# Calculate financial capability per vehicle
train_df['Income_Per_Car'] = train_df['Annual_Income_USD'] / (train_df['Number_of_Cars_Owned'] + 1)
test_df['Income_Per_Car'] = test_df['Annual_Income_USD'] / (test_df['Number_of_Cars_Owned'] + 1)

# Assess commute burden relative to income
train_df['Commute_to_Income_Ratio'] = train_df['Daily_Commute_km'] / (train_df['Annual_Income_USD'] + 1)
test_df['Commute_to_Income_Ratio'] = test_df['Daily_Commute_km'] / (test_df['Annual_Income_USD'] + 1)

print(f"New train shape: {train_df.shape}")
print(f"New test shape: {test_df.shape}")

New train shape: (668665, 18)
New test shape: (286571, 17)


In [8]:
# Suppress warnings for cleaner console output
warnings.filterwarnings("ignore")

# Redefine features including the newly engineered columns
X = train_df.drop(columns=['id', 'Will_Buy_EV'])
y = train_df['Will_Buy_EV']

# Setup Stratified K-Fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))

# Initialize LightGBM classifier
model = lgb.LGBMClassifier(n_estimators=500, random_state=42, n_jobs=-1)

print("Starting cross-validation training with engineered features...")

# Train model across folds
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Store validation predictions
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold + 1} ROC AUC: {fold_auc:.5f}")

# Calculate and display overall Out-Of-Fold score
oof_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall OOF ROC AUC with New Features: {oof_auc:.5f}")

Starting cross-validation training with engineered features...
[LightGBM] [Info] Number of positive: 93424, number of negative: 441508
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015705 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1163
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174646 -> initscore=-1.553048
[LightGBM] [Info] Start training from score -1.553048
Fold 1 ROC AUC: 0.93998
[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015718 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1163
[LightGBM] [Info] Num

## **6. XGBoost & Ensembling**

* Because our new features yielded a slight improvement, we will try to extract even more performance by introducing a different algorithm.
* We are going to train an `XGBoost` model using the exact same cross-validation strategy.
* Afterward, we plan to blend its predictions with our previous `LightGBM` results by taking a simple average.
* This ensembling technique aims to cancel out individual model errors and push our ROC AUC score higher.

In [9]:
# Initialize XGBoost classifier with explicit early stopping
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

xgb_oof_preds = np.zeros(len(train_df))

print("Starting XGBoost cross-validation training...")

# Train XGBoost model across folds
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Store validation predictions
    xgb_oof_preds[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, xgb_oof_preds[val_idx])
    print(f"Fold {fold + 1} ROC AUC: {fold_auc:.5f}")

# Calculate individual XGBoost Out-Of-Fold score
xgb_oof_auc = roc_auc_score(y, xgb_oof_preds)
print(f"\nOverall XGBoost OOF ROC AUC: {xgb_oof_auc:.5f}")

# Calculate blended predictions (LightGBM + XGBoost)
blended_oof_preds = (oof_preds + xgb_oof_preds) / 2
blended_auc = roc_auc_score(y, blended_oof_preds)
print(f"Blended Ensemble OOF ROC AUC: {blended_auc:.5f}")

Starting XGBoost cross-validation training...
Fold 1 ROC AUC: 0.94049
Fold 2 ROC AUC: 0.94151
Fold 3 ROC AUC: 0.94275
Fold 4 ROC AUC: 0.94230
Fold 5 ROC AUC: 0.94176

Overall XGBoost OOF ROC AUC: 0.94175
Blended Ensemble OOF ROC AUC: 0.94174


## **7. CatBoost & 3-Way Ensembling**

* Following the solid performance of our 2-way blend, we will introduce a third algorithm, `CatBoost`, known for its strong handling of categorical data.
* We are going to train this new model using the same cross-validation strategy to ensure perfectly comparable results.
* Once trained, we plan to create a 3-way ensemble by averaging the validation predictions from `LightGBM`, `XGBoost`, and `CatBoost`.
* This triple-blend approach aims to further reduce variance and maximize our final ROC AUC score before we consider hyperparameter tuning.

In [10]:
cat_file_path = '../models/catboost_model.pkl'
cat_preds_path = '../models/catboost_oof_preds.pkl'

# Check if both the serialized model and predictions exist on disk
if os.path.exists(cat_file_path) and os.path.exists(cat_preds_path):
    print("Loading existing CatBoost model and OOF predictions from local storage...")
    cat_model = joblib.load(cat_file_path)
    cat_oof_preds = joblib.load(cat_preds_path)
    print("CatBoost objects successfully loaded.")
else:
    print("Starting CatBoost cross-validation training...")
    cat_model = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6,
        random_seed=42, thread_count=-1, verbose=False
    )
    cat_oof_preds = np.zeros(len(train_df))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
        cat_oof_preds[val_idx] = cat_model.predict_proba(X_val)[:, 1]
        
    print(f"Overall CatBoost OOF ROC AUC: {roc_auc_score(y, cat_oof_preds):.5f}")
    
    # Save the trained model and predictions for future sessions
    joblib.dump(cat_model, cat_file_path)
    joblib.dump(cat_oof_preds, cat_preds_path)
    print("CatBoost model and OOF predictions successfully saved to disk.")

Loading existing CatBoost model and OOF predictions from local storage...
CatBoost objects successfully loaded.


## **8. Hyperparameter Optimization with Optuna (XGBoost)**

* We are going to define an objective function for `Optuna` to test different hyperparameter combinations on our `XGBoost` model.
* We will set up a search space for key parameters like `learning rate`, `max depth`, and `subsampling rates` to explore a wide range of configurations.
* We plan to run the optimization process for a fast 10-trial sprint to let the algorithm find an improved setup without taking hours to compute.
* Once finished, we will extract the best parameters found by `Optuna` to train our final, fully optimized model.

In [11]:
# Suppress Optuna logs to keep the console output clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Define the hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Setup Stratified K-Fold with 3 splits for faster evaluation during tuning
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    
    for train_idx, val_idx in skf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**param)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        
    # Return the target metric to maximize
    return roc_auc_score(y, oof_preds)

In [12]:
optuna_file_path = '../models/optuna_xgb_study.pkl'

# Check if the study already exists on disk
if os.path.exists(optuna_file_path):
    print("Loading existing Optuna study from local storage...")
    study = joblib.load(optuna_file_path)
else:
    print("Starting Optuna optimization for XGBoost (10 trials)...")
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)
    
    # Save the optimization study for future sessions
    joblib.dump(study, optuna_file_path)
    print("Optuna study successfully saved to disk.")

# Retrieve and display the best parameters and score
print(f"\nBest ROC AUC found: {study.best_value:.5f}")
print("Best parameters:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")

Loading existing Optuna study from local storage...

Best ROC AUC found: 0.94166
Best parameters:
    'n_estimators': 595,
    'max_depth': 5,
    'learning_rate': 0.040003118717712546,
    'subsample': 0.8421863523392037,
    'colsample_bytree': 0.6797870491228344,
    'min_child_weight': 2,


## **9. Advanced Feature Engineering (Target Encoding)**

* Following the baseline improvements from our custom features, we will now apply Target Encoding to our categorical variables to extract deeper statistical signals.

* We are going to calculate the mean probability of the target (`Will_Buy_EV`) for each category and create new columns with these probabilities.

* To prevent data leakage and overfitting, we plan to use a cross-validation approach within the encoder itself before passing the data to the model.

* Once these new probability features are added, we will retrain our XGBoost model to evaluate if this historical perspective pushes our ROC AUC higher.

In [13]:
# Define columns to target-encode based on previous categorical features
te_cols = [col + '_TE' for col in categorical_cols]

# Initialize TargetEncoder with cross-fitting to prevent data leakage
te = TargetEncoder(random_state=42)

# Fit the encoder on training data and create new probability columns
train_df[te_cols] = te.fit_transform(train_df[categorical_cols], train_df['Will_Buy_EV'])
test_df[te_cols] = te.transform(test_df[categorical_cols])

# Redefine feature matrix including the new target encoded columns
X = train_df.drop(columns=['id', 'Will_Buy_EV'])
y = train_df['Will_Buy_EV']

print(f"New feature count: {X.shape[1]}")
print("Starting XGBoost cross-validation with Target Encoded features...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_oof_preds_te = np.zeros(len(train_df))

# Initialize XGBoost with default baseline parameters for fair comparison
model_xgb_te = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05, 
    random_state=42, n_jobs=-1, early_stopping_rounds=50
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model_xgb_te.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    xgb_oof_preds_te[val_idx] = model_xgb_te.predict_proba(X_val)[:, 1]

xgb_te_auc = roc_auc_score(y, xgb_oof_preds_te)
print(f"XGBoost OOF ROC AUC with Target Encoding: {xgb_te_auc:.5f}")

New feature count: 22
Starting XGBoost cross-validation with Target Encoded features...
XGBoost OOF ROC AUC with Target Encoding: 0.93451


## **10. Feature Selection & K-Means Clustering**

* Since Target Encoding introduced noise and degraded our performance, we are going to drop those columns to return to our best-performing dataset.

* We will then introduce a `K-Means` clustering algorithm to group individuals based on their core continuous characteristics (like `Age`, `Income`, and `Commute`).

* We plan to add this cluster assignment as a new categorical feature, providing the tree models with a macro-level perspective of user profiles.

* Finally, we will retrain our `LightGBM` model on this clustered dataset to evaluate if this structural grouping helps push our score beyond the `0.9417` barrier.

In [14]:
warnings.filterwarnings("ignore")

# Drop the underperforming Target Encoded columns
train_df = train_df.drop(columns=te_cols)
test_df = test_df.drop(columns=te_cols)
X = train_df.drop(columns=['id', 'Will_Buy_EV'])
y = train_df['Will_Buy_EV']

print("Target Encoded columns dropped. Returning to previous feature space.")

# Select continuous features for clustering
cluster_features = ['Age', 'Annual_Income_USD', 'Daily_Commute_km']

# Standardize features before applying K-Means (distance-based algorithms require scaling)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[cluster_features])
test_scaled = scaler.transform(test_df[cluster_features])

# Initialize and fit K-Means to create 5 distinct customer personas
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
X['Customer_Persona'] = kmeans.fit_predict(train_scaled)
test_df['Customer_Persona'] = kmeans.predict(test_scaled)

# Ensure the new feature is treated as categorical by LightGBM
X['Customer_Persona'] = X['Customer_Persona'].astype('category')

print(f"New feature count with K-Means clusters: {X.shape[1]}")
print("Starting LightGBM cross-validation with clustered features...")

# Setup CV and LightGBM model (using baseline configuration for fair comparison)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_oof_preds_kmeans = np.zeros(len(X))

model_lgb_kmeans = lgb.LGBMClassifier(n_estimators=500, random_state=42, n_jobs=-1)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model_lgb_kmeans.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    lgb_oof_preds_kmeans[val_idx] = model_lgb_kmeans.predict_proba(X_val)[:, 1]
    
lgb_kmeans_auc = roc_auc_score(y, lgb_oof_preds_kmeans)
print(f"\nLightGBM OOF ROC AUC with Customer Personas: {lgb_kmeans_auc:.5f}")

Target Encoded columns dropped. Returning to previous feature space.
New feature count with K-Means clusters: 17
Starting LightGBM cross-validation with clustered features...
[LightGBM] [Info] Number of positive: 93424, number of negative: 441508
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023735 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1169
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174646 -> initscore=-1.553048
[LightGBM] [Info] Start training from score -1.553048
[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021959 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, yo

## **11. Environmental Analytics & Stress Profiling**

* We are going to drop the K-Means column and focus on hard domain logic by crossing environmental and logistical variables.

* We will engineer an `'Eco_Readiness_Score'` that mathematically combines a user's environmental concern with available subsidies.

* We also plan to calculate a `'Commute_Stress_Index'` by dividing the daily commute by the total available charging stations to quantify range anxiety.

* We will train our best-performing `XGBoost` baseline on this new environmental data configuration to check if we can finally break the 0.942 ceiling.

In [15]:
# Drop the underperforming K-Means column
X = X.drop(columns=['Customer_Persona'], errors='ignore')
test_df = test_df.drop(columns=['Customer_Persona'], errors='ignore')

# Engineer environmental and economic interaction features
X['Eco_Readiness'] = X['Environmental_Concern_Level'] * (X['Subsidy_Available'] + 1)
test_df['Eco_Readiness'] = test_df['Environmental_Concern_Level'] * (test_df['Subsidy_Available'] + 1)

# Calculate commute stress (adding 1 to avoid division by zero)
X['Commute_Stress_Index'] = X['Daily_Commute_km'] / (train_df['Total_Charging_Stations'] + 1)
test_df['Commute_Stress_Index'] = test_df['Daily_Commute_km'] / (test_df['Total_Charging_Stations'] + 1)

print("Environmental and stress features engineered. Starting XGBoost CV...")

xgb_oof_preds_eco = np.zeros(len(X))
model_xgb_eco = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05, 
    random_state=42, n_jobs=-1, early_stopping_rounds=50
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model_xgb_eco.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    xgb_oof_preds_eco[val_idx] = model_xgb_eco.predict_proba(X_val)[:, 1]

xgb_eco_auc = roc_auc_score(y, xgb_oof_preds_eco)
print(f"\nXGBoost OOF ROC AUC with Eco & Stress Features: {xgb_eco_auc:.5f}")

Environmental and stress features engineered. Starting XGBoost CV...

XGBoost OOF ROC AUC with Eco & Stress Features: 0.94174


## **12. Advanced Ensembling (Stacking)**

* We are going to move beyond simple blending and implement a `Stacking Classifier` to intelligently combine our baseline models.

* We will use the `Out-Of-Fold (OOF)` predictions from our `LightGBM`, `XGBoost`, and `CatBoost` models as the input features (Level 1) for a new meta-model.

* We plan to train a `Logistic Regression` algorithm as our meta-learner (Level 2) to calculate the optimal weight for each model's predictions based on their historical accuracy.

* This strategy allows the meta-model to learn and correct the individual blind spots of the base algorithms, aiming to push our final ROC AUC beyond the 0.9417 ceiling.

In [16]:
# Combine the OOF predictions from base models into a new feature matrix
# Note: Ensure oof_preds (LGBM), xgb_oof_preds, and cat_oof_preds are in memory
X_meta = np.column_stack((oof_preds, xgb_oof_preds, cat_oof_preds))

print("Starting Stacking CV with Logistic Regression meta-model...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meta_oof_preds = np.zeros(len(X_meta))

# Initialize the Level-2 meta-learner
meta_model = LogisticRegression(random_state=42, max_iter=1000)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta, y)):
    X_meta_train, y_train = X_meta[train_idx], y.iloc[train_idx]
    X_meta_val, y_val = X_meta[val_idx], y.iloc[val_idx]
    
    meta_model.fit(X_meta_train, y_train)
    meta_oof_preds[val_idx] = meta_model.predict_proba(X_meta_val)[:, 1]

stacking_auc = roc_auc_score(y, meta_oof_preds)
print(f"\nStacking Ensemble OOF ROC AUC: {stacking_auc:.5f}")

# Display the learned weights for LightGBM, XGBoost, and CatBoost respectively
print(f"Meta-model coefficients: {meta_model.coef_[0]}")

Starting Stacking CV with Logistic Regression meta-model...

Stacking Ensemble OOF ROC AUC: 0.94169
Meta-model coefficients: [2.26409391 2.30937124 2.25483806]


## **13. Deep Learning (Multi-Layer Perceptron)**

* We are going to build a neural network using TensorFlow and Keras to search for non-linear patterns that tree-based algorithms typically miss.

* We will completely standardize our feature matrix, as neural networks require all variables to be on the same scale to optimize effectively.

* We plan to design a Multi-Layer Perceptron (MLP) featuring dense layers, batch normalization, and dropout to strictly control overfitting.

* We will run this architecture through our cross-validation loop to generate new out-of-fold predictions and evaluate if it breaks our score ceiling.

In [19]:
nn_model_path = '../models/nn_model.keras'
nn_preds_path = '../models/nn_oof_preds.pkl'

# Neural networks require all inputs to be scaled uniformly
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check if both the serialized model and predictions exist on disk
if os.path.exists(nn_model_path) and os.path.exists(nn_preds_path):
    print("Loading existing Neural Network model and OOF predictions from local storage...")
    model = load_model(nn_model_path)
    nn_oof_preds = joblib.load(nn_preds_path)
    print("Neural Network objects successfully loaded.")
else:
    print("Starting Deep Learning cross-validation (Multi-Layer Perceptron)...")
    nn_oof_preds = np.zeros(len(X_scaled))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        X_train, y_train = X_scaled[train_idx], y.iloc[train_idx]
        X_val, y_val = X_scaled[val_idx], y.iloc[val_idx]
        
        # Build a sequential neural network architecture
        model = Sequential([
            Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
            BatchNormalization(),
            Dropout(0.3),
            Dense(32, activation='relu'),
            BatchNormalization(),
            Dropout(0.2),
            Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer='adam', 
            loss='binary_crossentropy', 
            metrics=[tf.keras.metrics.AUC(name='auc')]
        )
        
        early_stopping = EarlyStopping(
            monitor='val_auc', 
            mode='max', 
            patience=10, 
            restore_best_weights=True
        )
        
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=256,
            callbacks=[early_stopping],
            verbose=0
        )
        
        # Predict probabilities for the validation set
        nn_oof_preds[val_idx] = model.predict(X_val, verbose=0).flatten()
        
    print(f"\nNeural Network OOF ROC AUC: {roc_auc_score(y, nn_oof_preds):.5f}")
    
    # Save the final fold's model and the complete predictions array
    model.save(nn_model_path)
    joblib.dump(nn_oof_preds, nn_preds_path)
    print("Neural Network model and OOF predictions successfully saved to disk.")

Starting Deep Learning cross-validation (Multi-Layer Perceptron)...

Neural Network OOF ROC AUC: 0.93854
Neural Network model and OOF predictions successfully saved to disk.


## **14. Feature Importance & Selection**

* We are going to plot and evaluate the feature importances from our best `XGBoost` baseline.

* We will identify the lowest-contributing features that might be introducing noise rather than predictive power.

* We plan to drop these bottom features and retrain the model using our standard cross-validation.

* This pruning process aims to streamline the dataset and strictly isolate the variables that maximize our ROC AUC.

In [20]:
print("Fitting XGBoost to evaluate feature importance...")
importance_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05, 
    random_state=42, n_jobs=-1
)
importance_model.fit(X, y)

# Retrieve and sort feature importances
importances = importance_model.feature_importances_
feature_names = X.columns
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(feat_imp_df)

# Identify and drop the 3 least important features
features_to_drop = feat_imp_df.tail(3)['Feature'].tolist()
print(f"\nDropping least important features: {features_to_drop}")

X_pruned = X.drop(columns=features_to_drop)

# Cross-validate to check if the score improves without the noise
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_oof_preds_pruned = np.zeros(len(X_pruned))

print("Starting XGBoost cross-validation on pruned dataset...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_pruned, y)):
    X_train, y_train = X_pruned.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_pruned.iloc[val_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05, 
        random_state=42, n_jobs=-1, early_stopping_rounds=50
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    xgb_oof_preds_pruned[val_idx] = model.predict_proba(X_val)[:, 1]

pruned_auc = roc_auc_score(y, xgb_oof_preds_pruned)
print(f"\nPruned XGBoost OOF ROC AUC: {pruned_auc:.5f}")

Fitting XGBoost to evaluate feature importance...

Feature Importances:
                        Feature  Importance
16                Eco_Readiness    0.860652
6   Environmental_Concern_Level    0.064722
12          Range_Anxiety_Level    0.022914
1             Annual_Income_USD    0.016852
11            Subsidy_Available    0.012143
10       Home_Charging_Possible    0.004241
2              Daily_Commute_km    0.002544
15      Commute_to_Income_Ratio    0.002125
0                           Age    0.001917
4   Charging_Stations_Near_Home    0.001571
8                     City_Type    0.001537
14               Income_Per_Car    0.001448
3          Number_of_Cars_Owned    0.001370
17         Commute_Stress_Index    0.001249
5   Charging_Stations_Near_Work    0.001233
13      Total_Charging_Stations    0.001219
9              Current_Car_Type    0.001202
7                        Gender    0.001062

Dropping least important features: ['Total_Charging_Stations', 'Current_Car_Type', 'Gender'

## **15. Hyperparameter Tuning on Pruned Data (Optuna)**

* We are going to restart our Optuna optimization process specifically targeting the newly streamlined dataset.

* We will configure the objective function to evaluate XGBoost strictly on the features that proved to be highly predictive.

* We plan to run a fast batch of 20 trials to give the algorithm enough room to find a better configuration without exceeding the time limit.

* This tuning phase aims to squeeze out any remaining performance potential before we finally generate the competition submission.

In [21]:
optuna_pruned_path = '../models/optuna_xgb_pruned_study.pkl'

# Suppress Optuna logs to keep the console output clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_pruned(trial):
    # Define the hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 600),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Setup Stratified K-Fold with 3 splits for faster evaluation during tuning
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X_pruned))
    
    for train_idx, val_idx in skf.split(X_pruned, y):
        X_train, y_train = X_pruned.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_pruned.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**param)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        
    # Return the target metric to maximize
    return roc_auc_score(y, oof_preds)

# Check if the study already exists on disk
if os.path.exists(optuna_pruned_path):
    print("Loading existing pruned Optuna study from local storage...")
    study_pruned = joblib.load(optuna_pruned_path)
    print("Pruned Optuna study successfully loaded.")
else:
    print("Starting Optuna optimization on pruned dataset (20 trials)...")
    # Create a study object and optimize the objective function
    study_pruned = optuna.create_study(direction='maximize')
    study_pruned.optimize(objective_pruned, n_trials=20)
    
    # Save the optimization study for future sessions
    joblib.dump(study_pruned, optuna_pruned_path)
    print("Pruned Optuna study successfully saved to disk.")

print(f"\nBest ROC AUC found: {study_pruned.best_value:.5f}")
print("Best parameters:")
for key, value in study_pruned.best_params.items():
    print(f"    '{key}': {value},")

Starting Optuna optimization on pruned dataset (20 trials)...
Pruned Optuna study successfully saved to disk.

Best ROC AUC found: 0.94183
Best parameters:
    'n_estimators': 598,
    'max_depth': 4,
    'learning_rate': 0.09079236261070038,
    'subsample': 0.9940964342870352,
    'colsample_bytree': 0.7186365920296764,
    'min_child_weight': 1,


## **16. Final Submission with Optimized XGBoost**

* We are going to train the final model using the exact optimal hyperparameters discovered by Optuna.

* We will fit this model on the entire pruned training dataset to maximize data exposure.

* We plan to generate predictions for the aligned test set and export the final CSV file for Kaggle.

In [22]:
print("Training final XGBoost model with Optuna best parameters...")

# Initialize the model using the optimal parameters found
final_optimized_xgb = xgb.XGBClassifier(
    **study_pruned.best_params,
    random_state=42, 
    n_jobs=-1
)

# Train on the complete pruned dataset
final_optimized_xgb.fit(X_pruned, y)

print("Generating predictions for the test set...")

# Ensure test set is pruned identically to the training set
features_to_drop = ['Total_Charging_Stations', 'Current_Car_Type', 'Gender']
X_test_pruned = test_df.drop(columns=features_to_drop + ['id'], errors='ignore')

# Predict probabilities
final_test_preds = final_optimized_xgb.predict_proba(X_test_pruned)[:, 1]

# Format predictions to match Kaggle submission requirements
submission = pd.DataFrame({
    'id': test_df['id'],
    'Will_Buy_EV': final_test_preds
})

# Export the submission file to the root directory
submission_path = '../submission_day1_optimized.csv'
submission.to_csv(submission_path, index=False)
print(f"Submission successfully saved to {submission_path}")

Training final XGBoost model with Optuna best parameters...
Generating predictions for the test set...
Submission successfully saved to ../submission_day1_optimized.csv
